# Universal Tabular Hackathon Preset

This notebook is a reusable starter for Kaggle-style tabular competitions:

- Reads `train.csv`, `test.csv`, and `sample_submission.csv`
- Infers ID, target, task type, and submission schema
- Audits missing values, cardinality, leakage-looking columns, and class balance
- Builds strong scikit-learn baselines with preprocessing pipelines
- Cross-validates, blends useful models, and writes a valid submission file

Preset dataset notes from the supplied files:

- Train: `223,084` rows, `20` columns
- Test: `74,361` rows, `19` columns
- Target: `History of HeartDisease or Attack`
- ID column: `ID`
- Target labels: `No` / `Yes`, with about `8.1%` positive among all training rows
- `1,694` train rows have missing target labels and are excluded from supervised training
- `sample_submission.csv` has the same IDs as `test.csv`

In [3]:
# ============================================================
# 0. Configuration
# ============================================================
from pathlib import Path
import os
import re
import json
import math
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42
N_SPLITS = 5
FAST_MODE = False      # set True for quick smoke tests
USE_OPTIONAL_GBM = True  # uses lightgbm/catboost/xgboost if already installed

# Change only this block for another hackathon.
DATA_DIR = Path(r".")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"
OUTPUT_PATH = Path("submission.csv")

assert TRAIN_PATH.exists(), TRAIN_PATH
assert TEST_PATH.exists(), TEST_PATH
assert SAMPLE_SUB_PATH.exists(), SAMPLE_SUB_PATH

In [6]:
# ============================================================
# 1. Load data and infer competition schema
# ============================================================
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("train:", train_raw.shape)
print("test :", test_raw.shape)
print("sample_submission:", sample_sub.shape)

ID_COL = "ID"
TARGET_COL = "History of HeartDisease or Attack" 
SUBMISSION_TARGET_COL = "History of HeartDisease or Attack"

print("ID_COL:", ID_COL)
print("TARGET_COL:", TARGET_COL)
print("SUBMISSION_TARGET_COL:", SUBMISSION_TARGET_COL)
print("sample IDs match test:", sample_sub[ID_COL].equals(test_raw[ID_COL]) if ID_COL in sample_sub and ID_COL in test_raw else "unknown")

train: (223084, 20)
test : (74361, 19)
sample_submission: (74361, 2)
ID_COL: ID
TARGET_COL: History of HeartDisease or Attack
SUBMISSION_TARGET_COL: History of HeartDisease or Attack
sample IDs match test: True


In [7]:
# ============================================================
# 2. Compact EDA and data audit
# ============================================================
display(train_raw.head())
display(test_raw.head())

audit = pd.DataFrame({
    "column": train_raw.columns,
    "dtype": train_raw.dtypes.astype(str).values,
    "train_missing": train_raw.isna().sum().values,
    "train_missing_pct": (train_raw.isna().mean().values * 100).round(2),
    "train_nunique": train_raw.nunique(dropna=False).values,
})

test_audit = pd.DataFrame({
    "column": test_raw.columns,
    "test_missing": test_raw.isna().sum().values,
    "test_missing_pct": (test_raw.isna().mean().values * 100).round(2),
    "test_nunique": test_raw.nunique(dropna=False).values,
})

audit = audit.merge(test_audit, on="column", how="left")
display(audit.sort_values(["train_missing_pct", "train_nunique"], ascending=False))

print("Target distribution including missing:")
display(train_raw[TARGET_COL].value_counts(dropna=False).to_frame("count"))

print("Duplicate IDs:", {
    "train": int(train_raw[ID_COL].duplicated().sum()) if ID_COL in train_raw else None,
    "test": int(test_raw[ID_COL].duplicated().sum()) if ID_COL in test_raw else None,
})

cat_diff_rows = []
for c in [c for c in test_raw.columns if c in train_raw.columns and c != ID_COL]:
    if train_raw[c].dtype == "object" or test_raw[c].dtype == "object":
        train_vals = set(train_raw[c].dropna().astype(str).unique())
        test_vals = set(test_raw[c].dropna().astype(str).unique())
        if train_vals - test_vals or test_vals - train_vals:
            cat_diff_rows.append({
                "column": c,
                "train_only": sorted(train_vals - test_vals)[:10],
                "test_only": sorted(test_vals - train_vals)[:10],
            })
display(pd.DataFrame(cat_diff_rows) if cat_diff_rows else pd.DataFrame({"message": ["No train/test categorical level mismatch found."]}))

,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes


,ID,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,test_000001,Yes,Yes,Yes,24.84,No,No,No,Yes,No,Yes,No,Good,No,Female,Some college or technical school,"$20,000 to less than $25,000",71,Yes
1,test_000002,Yes,No,Yes,29.08,Yes,No,No,No,No,Yes,No,Fair,No,Female,College graduate,"$50,000 to less than $75,000",61,No
2,test_000003,Yes,Yes,Yes,35.23,Yes,No,No,No,No,Yes,No,Fair,Yes,Female,Some college or technical school,"Less than $10,000",67,Yes
3,test_000004,No,No,Yes,24.78,Yes,No,No,No,No,Yes,No,Fair,No,Female,Some college or technical school,"$50,000 to less than $75,000",50,Yes
4,test_000005,No,No,Yes,27.57,Yes,No,No,No,No,Yes,No,Fair,No,Male,Some college or technical school,"$25,000 to less than $35,000",40,Yes


,column,dtype,train_missing,train_missing_pct,train_nunique,test_missing,test_missing_pct,test_nunique
3,Told High Cholesterol,object,32186,14.43,3,0.0,0.0,2.0
5,Body Mass Index,float64,11782,5.28,4908,0.0,0.0,3921.0
1,History of HeartDisease or Attack,object,1694,0.76,3,NaN,NaN,NaN
0,ID,object,0,0.00,223084,0.0,0.0,74361.0
18,Age,int64,0,0.00,83,0.0,0.0,83.0
17,Income Level,object,0,0.00,8,0.0,0.0,8.0
13,General Health,object,1,0.00,6,0.0,0.0,5.0
16,Education Level,object,0,0.00,6,0.0,0.0,6.0
6,Smoked 100+ Cigarettes,object,1,0.00,3,0.0,0.0,2.0
8,Diagnosed Diabetes,object,3,0.00,3,0.0,0.0,2.0


Target distribution including missing:


,count
History of HeartDisease or Attack,
No,203322
Yes,18068
NaN,1694


Duplicate IDs: {'train': 0, 'test': 0}


,message
0,No train/test categorical level mismatch found.


In [8]:
# ============================================================
# 3. Prepare features and infer task type
# ============================================================
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, log_loss,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier, ExtraTreesRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor

train = train_raw.copy()
test = test_raw.copy()

# Rows with missing target cannot train supervised models.
missing_target_mask = train[TARGET_COL].isna()
if missing_target_mask.any():
    print(f"Dropping {missing_target_mask.sum():,} rows with missing target from training.")
    train = train.loc[~missing_target_mask].reset_index(drop=True)

drop_cols = [c for c in [ID_COL, TARGET_COL] if c in train.columns]
X = train.drop(columns=drop_cols)
y_raw = train[TARGET_COL]
X_test = test.drop(columns=[c for c in [ID_COL, TARGET_COL] if c in test.columns])

# Align test feature columns exactly to training feature columns.
missing_in_test = [c for c in X.columns if c not in X_test.columns]
extra_in_test = [c for c in X_test.columns if c not in X.columns]
for c in missing_in_test:
    X_test[c] = np.nan
X_test = X_test[X.columns]
if extra_in_test:
    print("Ignored extra test columns:", extra_in_test)

def infer_task_type(y):
    if pd.api.types.is_numeric_dtype(y):
        nunique = y.nunique(dropna=True)
        if nunique <= 20 and nunique / len(y) < 0.05:
            return "classification"
        return "regression"
    return "classification"

TASK = infer_task_type(y_raw)
print("TASK:", TASK)

if TASK == "classification":
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)
    class_names = list(label_encoder.classes_)
    n_classes = len(class_names)
    positive_label = class_names[-1] if n_classes == 2 else None
    print("classes:", dict(enumerate(class_names)))
    print(pd.Series(y_raw).value_counts(normalize=True).rename("ratio"))
else:
    label_encoder = None
    y = y_raw.astype(float).values
    n_classes = None

num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"numeric columns: {len(num_cols)}")
print(f"categorical columns: {len(cat_cols)}")
print("categorical:", cat_cols[:30])

Dropping 1,694 rows with missing target from training.
TASK: classification
classes: {0: 'No', 1: 'Yes'}
History of HeartDisease or Attack
No     0.918388
Yes    0.081612
Name: ratio, dtype: float64
numeric columns: 2
categorical columns: 16
categorical: ['High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked', 'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Leisure Physical Activity', 'Heavy Alcohol Consumption', 'Health Care Coverage', 'Doctor Visit Cost Barrier', 'General Health', 'Difficulty Walking', 'Sex', 'Education Level', 'Income Level', 'Vegetable or Fruit Intake (1+ per Day)']


In [9]:
# ============================================================
# 4. Preprocessing and model zoo
# ============================================================
def make_onehot_preprocessor():
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", ohe),
            ]), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def make_ordinal_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ]), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


models = {}

if TASK == "classification":
    models["logreg_ohe"] = Pipeline([
        ("prep", make_onehot_preprocessor()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
    ])
    models["hgb_ordinal"] = Pipeline([
        ("prep", make_ordinal_preprocessor()),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_iter=150 if FAST_MODE else 350,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )),
    ])
    models["extra_trees_ordinal"] = Pipeline([
        ("prep", make_ordinal_preprocessor()),
        ("model", ExtraTreesClassifier(
            n_estimators=150 if FAST_MODE else 500,
            min_samples_leaf=4,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])
else:
    models["ridge_ohe"] = Pipeline([
        ("prep", make_onehot_preprocessor()),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ])
    models["hgb_ordinal"] = Pipeline([
        ("prep", make_ordinal_preprocessor()),
        ("model", HistGradientBoostingRegressor(
            learning_rate=0.06,
            max_iter=150 if FAST_MODE else 350,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )),
    ])
    models["extra_trees_ordinal"] = Pipeline([
        ("prep", make_ordinal_preprocessor()),
        ("model", ExtraTreesRegressor(
            n_estimators=150 if FAST_MODE else 500,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


# Optional installed libraries. No install required; skipped automatically if unavailable.
if USE_OPTIONAL_GBM:
    try:
        from lightgbm import LGBMClassifier, LGBMRegressor
        if TASK == "classification":
            models["lightgbm_ordinal"] = Pipeline([
                ("prep", make_ordinal_preprocessor()),
                ("model", LGBMClassifier(
                    n_estimators=500 if FAST_MODE else 1500,
                    learning_rate=0.03,
                    num_leaves=31,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    class_weight="balanced" if n_classes == 2 else None,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ])
        else:
            models["lightgbm_ordinal"] = Pipeline([
                ("prep", make_ordinal_preprocessor()),
                ("model", LGBMRegressor(
                    n_estimators=500 if FAST_MODE else 1500,
                    learning_rate=0.03,
                    num_leaves=31,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ])
    except Exception as e:
        print("LightGBM skipped:", type(e).__name__)

print("models:", list(models))

models: ['logreg_ohe', 'hgb_ordinal', 'extra_trees_ordinal', 'lightgbm_ordinal']


In [10]:
# ============================================================
# 5. Cross-validation
# ============================================================
def classification_metrics(y_true, pred, proba=None):
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "f1_macro": f1_score(y_true, pred, average="macro"),
    }
    if proba is not None:
        try:
            if proba.ndim == 2 and proba.shape[1] == 2:
                out["roc_auc"] = roc_auc_score(y_true, proba[:, 1])
                out["log_loss"] = log_loss(y_true, proba)
            elif proba.ndim == 2 and proba.shape[1] > 2:
                out["roc_auc_ovr"] = roc_auc_score(y_true, proba, multi_class="ovr")
                out["log_loss"] = log_loss(y_true, proba)
        except Exception:
            pass
    return out


def regression_metrics(y_true, pred):
    rmse = mean_squared_error(y_true, pred, squared=False)
    return {
        "rmse": rmse,
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred),
    }


if TASK == "classification":
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
else:
    splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
oof_store = {}

for model_name, model in models.items():
    print(f"\n=== {model_name} ===")
    if TASK == "classification":
        oof_pred = np.zeros(len(X), dtype=int)
        oof_proba = np.zeros((len(X), n_classes), dtype=float)
    else:
        oof_pred = np.zeros(len(X), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)

        if TASK == "classification":
            oof_pred[va_idx] = pred
            if hasattr(model, "predict_proba"):
                proba = model.predict_proba(X_va)
                oof_proba[va_idx, :proba.shape[1]] = proba
            else:
                proba = None
            metrics = classification_metrics(y_va, pred, proba)
        else:
            oof_pred[va_idx] = pred
            metrics = regression_metrics(y_va, pred)

        print("fold", fold, {k: round(v, 5) for k, v in metrics.items()})
        cv_rows.append({"model": model_name, "fold": fold, **metrics})

    if TASK == "classification":
        oof_store[model_name] = {"pred": oof_pred, "proba": oof_proba}
        full_metrics = classification_metrics(y, oof_pred, oof_proba)
    else:
        oof_store[model_name] = {"pred": oof_pred}
        full_metrics = regression_metrics(y, oof_pred)

    print("OOF", {k: round(v, 5) for k, v in full_metrics.items()})

cv = pd.DataFrame(cv_rows)
display(cv.groupby("model").mean(numeric_only=True).sort_values(
    "roc_auc" if "roc_auc" in cv.columns else ("rmse" if TASK == "regression" else "balanced_accuracy"),
    ascending=(TASK == "regression")
))


=== logreg_ohe ===
fold 1 {'accuracy': 0.76661, 'balanced_accuracy': 0.78392, 'f1_macro': 0.60866, 'roc_auc': 0.85998, 'log_loss': 0.4776}
fold 2 {'accuracy': 0.76851, 'balanced_accuracy': 0.78394, 'f1_macro': 0.60996, 'roc_auc': 0.85904, 'log_loss': 0.47757}
fold 3 {'accuracy': 0.7667, 'balanced_accuracy': 0.78613, 'f1_macro': 0.60942, 'roc_auc': 0.86245, 'log_loss': 0.4799}
fold 4 {'accuracy': 0.7662, 'balanced_accuracy': 0.7817, 'f1_macro': 0.60775, 'roc_auc': 0.85856, 'log_loss': 0.47893}
fold 5 {'accuracy': 0.76528, 'balanced_accuracy': 0.77981, 'f1_macro': 0.60656, 'roc_auc': 0.86029, 'log_loss': 0.47987}
OOF {'accuracy': 0.76666, 'balanced_accuracy': 0.7831, 'f1_macro': 0.60847, 'roc_auc': 0.86006, 'log_loss': 0.47878}

=== hgb_ordinal ===
fold 1 {'accuracy': 0.91998, 'balanced_accuracy': 0.54008, 'f1_macro': 0.55368, 'roc_auc': 0.86142, 'log_loss': 0.21013}
fold 2 {'accuracy': 0.92052, 'balanced_accuracy': 0.54655, 'f1_macro': 0.56406, 'roc_auc': 0.86196, 'log_loss': 0.20975}


,fold,accuracy,balanced_accuracy,f1_macro,roc_auc,log_loss
model,,,,,,
hgb_ordinal,3.0,0.920182,0.541823,0.556496,0.862032,0.209710
logreg_ohe,3.0,0.766661,0.783100,0.608471,0.860064,0.478776
lightgbm_ordinal,3.0,0.771959,0.779782,0.611084,0.858083,0.433657
extra_trees_ordinal,3.0,0.804463,0.766918,0.630062,0.853272,0.391221


In [11]:
# ============================================================
# 6. Choose models and tune threshold for imbalanced binary classification
# ============================================================
summary = cv.groupby("model").mean(numeric_only=True)
if TASK == "classification":
    if "roc_auc" in summary.columns:
        ranked_models = summary.sort_values("roc_auc", ascending=False).index.tolist()
    else:
        ranked_models = summary.sort_values("balanced_accuracy", ascending=False).index.tolist()
else:
    ranked_models = summary.sort_values("rmse", ascending=True).index.tolist()

TOP_MODELS = ranked_models[: min(3, len(ranked_models))]
print("Selected for blend:", TOP_MODELS)

best_threshold = 0.5
if TASK == "classification" and n_classes == 2:
    blend_oof_proba = np.mean([oof_store[m]["proba"] for m in TOP_MODELS], axis=0)
    scores = []
    for thr in np.linspace(0.05, 0.95, 91):
        pred_thr = (blend_oof_proba[:, 1] >= thr).astype(int)
        scores.append((thr, f1_score(y, pred_thr), balanced_accuracy_score(y, pred_thr)))
    thr_df = pd.DataFrame(scores, columns=["threshold", "f1", "balanced_accuracy"])
    display(thr_df.sort_values("balanced_accuracy", ascending=False).head(10))
    best_threshold = float(thr_df.sort_values("balanced_accuracy", ascending=False).iloc[0]["threshold"])
    print("best_threshold_for_balanced_accuracy:", best_threshold)

Selected for blend: ['hgb_ordinal', 'logreg_ohe', 'lightgbm_ordinal']


,threshold,f1,balanced_accuracy
27,0.32,0.345252,0.786341
26,0.31,0.341000,0.786291
29,0.34,0.353729,0.786200
30,0.35,0.358011,0.786038
25,0.30,0.336893,0.785774
28,0.33,0.348930,0.785773
31,0.36,0.361997,0.785256
24,0.29,0.332153,0.784406
32,0.37,0.365612,0.783842
33,0.38,0.369625,0.782724


best_threshold_for_balanced_accuracy: 0.31999999999999995


In [14]:
# ============================================================
# 7. Fit on all data, predict test, and create submission
# ============================================================
test_preds = []
fitted_models = {}

for model_name in TOP_MODELS:
    print("fit full:", model_name)
    model = models[model_name]
    model.fit(X, y)
    fitted_models[model_name] = model

    if TASK == "classification":
        if hasattr(model, "predict_proba"):
            test_preds.append(model.predict_proba(X_test))
        else:
            pred = model.predict(X_test)
            onehot = np.zeros((len(pred), n_classes))
            onehot[np.arange(len(pred)), pred] = 1
            test_preds.append(onehot)
    else:
        test_preds.append(model.predict(X_test))

submission = sample_sub.copy()

if TASK == "classification":
    blend_test_proba = np.mean(test_preds, axis=0)
    if n_classes == 2:
        pred_labels_encoded = (blend_test_proba[:, 1] >= best_threshold).astype(int)
    else:
        pred_labels_encoded = np.argmax(blend_test_proba, axis=1)
    pred_labels = label_encoder.inverse_transform(pred_labels_encoded)
    submission[SUBMISSION_TARGET_COL] = pred_labels
else:
    blend_test_pred = np.mean(test_preds, axis=0)
    submission[SUBMISSION_TARGET_COL] = blend_test_pred

display(submission.head())
print(submission[SUBMISSION_TARGET_COL].value_counts(dropna=False).head(20) if TASK == "classification" else submission[SUBMISSION_TARGET_COL].describe())

submission.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH.resolve())

fit full: hgb_ordinal
fit full: logreg_ohe
fit full: lightgbm_ordinal
[LightGBM] [Info] Number of positive: 18068, number of negative: 203322
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022777 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 390
[LightGBM] [Info] Number of data points in the train set: 221390, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


,ID,History of HeartDisease or Attack
0,test_000001,Yes
1,test_000002,No
2,test_000003,Yes
3,test_000004,No
4,test_000005,No


History of HeartDisease or Attack
No     46753
Yes    27608
Name: count, dtype: int64
Saved: C:\Users\ZBook\Downloads\super-ai-engineer-ss-6-heart-disease-prediction\submission.csv


In [15]:
# ============================================================
# 8. Optional: Feature importance / model interpretation
# ============================================================
def get_feature_names(preprocessor):
    try:
        return preprocessor.get_feature_names_out()
    except Exception:
        return np.array(X.columns)

importance_frames = []
for model_name, pipe in fitted_models.items():
    estimator = pipe.named_steps.get("model")
    prep = pipe.named_steps.get("prep")
    if hasattr(estimator, "feature_importances_"):
        names = get_feature_names(prep)
        imp = pd.DataFrame({
            "model": model_name,
            "feature": names[: len(estimator.feature_importances_)],
            "importance": estimator.feature_importances_,
        })
        importance_frames.append(imp)
    elif hasattr(estimator, "coef_"):
        names = get_feature_names(prep)
        coef = np.ravel(estimator.coef_)
        imp = pd.DataFrame({
            "model": model_name,
            "feature": names[: len(coef)],
            "importance": np.abs(coef),
        })
        importance_frames.append(imp)

if importance_frames:
    importance = pd.concat(importance_frames, ignore_index=True)
    display(importance.groupby("feature")["importance"].mean().sort_values(ascending=False).head(30).to_frame())
else:
    print("No built-in feature importance available for selected models.")

,importance
feature,
Body Mass Index,5140.500117
Age,5057.023067
Income Level,4629.000000
General Health,3367.000000
Education Level,2776.000000
Told High Cholesterol,1870.000000
Sex,1533.000000
Smoked 100+ Cigarettes,1431.000000
High Blood Pressure,1400.000000


## Fast improvement checklist

Use this after the first valid submission:

1. Confirm the competition metric and change the model ranking cell to optimize that metric.
2. Try target-specific thresholding only if the submission expects class labels and the metric rewards recall/balanced accuracy/F1.
3. Add domain features in a new cell before preprocessing, then rerun CV.
4. Use stronger installed models such as LightGBM, XGBoost, or CatBoost when allowed.
5. Compare public leaderboard movement with CV. If they disagree badly, investigate leakage, groups, time splits, or distribution shift.
6. Keep every submission tied to a CV score and notebook version.